# Homework | RAG

In this homework, we build a RAG system from scratch and then make it agentic - the same path as the module.

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
files[:3]

[RawRepositoryFile(filename='01-agentic-rag/lessons/01-intro.md', content='# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are

## Q1. How many lesson pages

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
documents.__len__()

72

In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

## Q2. Indexing and searching

In [ ]:
from rag_helper_files import RAGBase
from minsearch import Index

In [7]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

question =  "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    num_results=5
)

search_results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

## Q3. RAG

In [8]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [9]:
assistant = RAGBase(index, openai_client)

In [10]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

- Usage : ResponseUsage(input_tokens=7036, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=110, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7146)



'The agentic loop keeps calling the model inside a `while True` loop.\n\nEach iteration:\n1. It sends the full message history to the model.\n2. It checks the response for any `function_call` items.\n3. If there are function calls, it runs them, appends the results to `messages`, and loops again.\n4. If there are no function calls, it `break`s.\n\nSo the stop condition is simple: **when `has_function_calls` is `False`, the loop ends**.'

## Q4. Chunking

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
chunks.__len__()

295

In [13]:
chunks[:3]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

## Q5. RAG with chunking


In [14]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(documents)

question =  "How does the agentic loop keep calling the model until it stops?"

search_results = chunk_index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    num_results=5
)

search_results[0]

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [15]:
new_assistant_chunk = RAGBase(chunk_index, openai_client)

In [16]:
new_assistant_chunk.rag('How does the agentic loop keep calling the model until it stops?')

- Usage : ResponseUsage(input_tokens=7036, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7133)



'It keeps calling the model inside a `while True` loop and checks whether the response contains any `function_call` items.\n\n- If the model returns a function call, the code runs the tool, appends the tool output to `messages`, and loops again.\n- If the model returns only a normal message, `has_function_calls` stays `False`, and the loop breaks.\n\nSo the stop condition is: **no function calls in the latest response**.'

## Q6. Turning it into an agent

In [17]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [18]:
def search_with_agent(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
    )

In [28]:
search_tool = {
    "type": "function",
    'name': 'search_with_agent',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [20]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""
question = "How does the agentic loop work, and how is it different from plain RAG?"

In [29]:
agent_tools = Tools()
agent_tools.add_tool(search_with_agent, search_tool)

In [30]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search_with_agent',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [23]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [31]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [32]:
result = runner.loop(
    prompt=question,
    callback=callback,
)

-> Response received


-> Response received


In [34]:
result.cost

CostInfo(input_cost=Decimal('0.015156'), output_cost=Decimal('0.0018675'), total_cost=Decimal('0.0170235'))

In [33]:
result.tokens

TokenUsage(model='gpt-5.4-mini', input_tokens=20208, output_tokens=415)